# Pass Labelling — Line-Breaking Detection

**Question:** Does `turf.efi.label` correctly identify whether a pass crossed one or more
defensive unit lines, and does it agree with the precomputed annotations?

**Data:** FIFA WC 2022 — Brazil vs South Korea (match 10507) + precomputed `lbp_annotation_pff_wc22.csv`  
**Module under test:** `turf.efi.label`

> **Requires:** `data/pff/wc2022/` populated. Run with `uv sync --group viz` for plots.

In [ ]:
import pathlib
import tempfile

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from turf.efi.label import label_event
from turf.efi.line import (
    analyze_lines,
    fix_player_assignments,
    vote_line_count,
)
from turf.efi.viz import make_pitch, plot_lines_frame
from turf.io.pff import load_match

%matplotlib inline
matplotlib.rcParams['figure.facecolor'] = '#ffffff'

np.random.seed(42)

RAW_DIR     = pathlib.Path('../../data/raw/FIFA_WC_2022')
RESULTS_DIR = pathlib.Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

MATCH_ID  = 10507
HOME_TEAM = 'home'   # Brazil
AWAY_TEAM = 'away'   # South Korea

## Load data

In [ ]:
match    = load_match(RAW_DIR, MATCH_ID)
tracking = match.tracking
events   = match.events

# Build annotation table from events: one row per pass, with is_line_breaking derived
# from linesBrokenType (None or 'NONE' = not line-breaking; anything else = breaking)
passes = events[events['possession_type'] == 'PA'].copy().reset_index(drop=True)
passes['is_line_breaking'] = (
    passes['lines_broken'].notna() & (passes['lines_broken'] != 'NONE')
)
passes['event_idx'] = passes.index   # stable row id for build_lines_for_event

annot = passes

n_lbp   = annot['is_line_breaking'].sum()
n_total = len(annot)
pct     = n_lbp / n_total * 100
print(f'Match {MATCH_ID}: {n_total} passes, {n_lbp} line-breaking ({pct:.1f}%)')

## Label distribution

In [ ]:
by_team = annot.groupby(['is_home', 'is_line_breaking']).size().unstack(fill_value=0)
by_team.columns = ['not_breaking', 'line_breaking']
by_team['pct_lbp'] = (
    by_team['line_breaking'] / by_team.sum(axis=1) * 100
).round(1)
by_team.index = [
    f'{"home" if idx else "away"}' for idx in by_team.index
]
print(by_team.to_string())

## Helper: extract clip and run pipeline

In [ ]:
def build_lines_for_event(
    event_idx: int,
) -> tuple[pd.DataFrame, str, str] | None:
    """Extract tracking clip and run analyze_lines for an annotated pass.

    Returns (lines_df, defending_team, attacking_team) or None if clip is empty.
    """
    row = annot[annot['event_idx'] == event_idx]
    if row.empty:
        return None
    row = row.iloc[0]

    is_home     = bool(row['is_home'])
    attacking_team = HOME_TEAM if is_home else AWAY_TEAM
    defending_team = AWAY_TEAM if is_home else HOME_TEAM

    start_t = float(row['start_time'])
    end_t   = float(row['end_time'])
    if end_t <= start_t:
        end_t = start_t + 1.0
    period  = int(row['period'])

    clip = tracking[
        (tracking['period'] == period) &
        (tracking['time_s'] >= start_t) &
        (tracking['time_s'] <= end_t)
    ].copy().reset_index(drop=True)

    if len(clip) < 2:
        return None

    atk_dir = match.directions.for_team(defending_team, period)

    first_pass = analyze_lines(clip, defending_team, atk_dir)
    k = vote_line_count(first_pass)
    lines_df = fix_player_assignments(
        analyze_lines(clip, defending_team, atk_dir, force_n_lines=k),
        defending_team,
    )
    return lines_df, defending_team, attacking_team

## Example 1 — Non-line-breaking pass

In [ ]:
nlbp_rows = annot[
    (~annot['is_line_breaking']) &
    (annot['pass_outcome'] == 'success')
].head(5)

result_nlbp = None
for _, cand in nlbp_rows.iterrows():
    r = build_lines_for_event(int(cand['event_idx']))
    if r is not None:
        nlbp_lines_df, nlbp_def, nlbp_atk = r
        nlbp_annot_row = cand
        result_nlbp = r
        break

assert result_nlbp is not None, 'No valid non-LBP clip found'
print(f'Event idx: {nlbp_annot_row["event_idx"]}')
print(f'Pass: {nlbp_annot_row["passer_name"]} → {nlbp_annot_row["receiver_name"]}')
print(f'Precomputed is_line_breaking: {nlbp_annot_row["is_line_breaking"]}')

with tempfile.TemporaryDirectory() as tmpdir:
    event_dir = pathlib.Path(tmpdir)
    nlbp_lines_df.to_csv(event_dir / 'lines.csv', index=False)
    nlbp_result = label_event(event_dir, nlbp_def)

print(f'label_event result: {nlbp_result}')

In [ ]:
pitch, theme = make_pitch()
fig, ax = pitch.draw(figsize=(12, 8))
fig.patch.set_facecolor(theme['bg_color'])
ax.set_ylim(-42, 34)

plot_lines_frame(
    ax, nlbp_lines_df.iloc[0],
    defending_team=nlbp_def, attacking_team=nlbp_atk,
    theme=theme,
)

# Draw pass arrow: start ball → end ball
sb_x = float(nlbp_lines_df.iloc[0]['ball_x'])
sb_y = float(nlbp_lines_df.iloc[0]['ball_y'])
eb_x = float(nlbp_lines_df.iloc[-1]['ball_x'])
eb_y = float(nlbp_lines_df.iloc[-1]['ball_y'])
ax.annotate('', xy=(eb_x, eb_y), xytext=(sb_x, sb_y),
            arrowprops=dict(arrowstyle='->, head_width=0.4, head_length=0.3',
                           color=theme['text_color'], lw=2.5), zorder=8)

ax.set_title(
    f'Non-line-breaking pass — is_line_breaking={nlbp_result["is_line_breaking"]}',
    color=theme['text_color'], fontsize=11, pad=8
)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'nlbp_example.png', dpi=120, bbox_inches='tight')
plt.show()

## Example 2 — Line-breaking pass

In [ ]:
lbp_rows = annot[
    annot['is_line_breaking'] &
    (annot['pass_outcome'] == 'success')
].head(5)

result_lbp = None
for _, cand in lbp_rows.iterrows():
    r = build_lines_for_event(int(cand['event_idx']))
    if r is not None:
        lbp_lines_df, lbp_def, lbp_atk = r
        lbp_annot_row = cand
        result_lbp = r
        break

assert result_lbp is not None, 'No valid LBP clip found'
print(f'Event idx: {lbp_annot_row["event_idx"]}')
print(f'Pass: {lbp_annot_row["passer_name"]} → {lbp_annot_row["receiver_name"]}')
lb = lbp_annot_row["lines_broken"]
is_lb = lbp_annot_row["is_line_breaking"]
print(f'lines_broken: {lb}  is_line_breaking: {is_lb}')

with tempfile.TemporaryDirectory() as tmpdir:
    event_dir = pathlib.Path(tmpdir)
    lbp_lines_df.to_csv(event_dir / 'lines.csv', index=False)
    lbp_result = label_event(event_dir, lbp_def)

print(f'label_event result: {lbp_result}')

In [ ]:
pitch, theme = make_pitch()
fig, ax = pitch.draw(figsize=(12, 8))
fig.patch.set_facecolor(theme['bg_color'])
ax.set_ylim(-42, 34)

plot_lines_frame(
    ax, lbp_lines_df.iloc[0],
    defending_team=lbp_def, attacking_team=lbp_atk,
    theme=theme,
)

# Pass arrow (gold to signal line-breaking)
sb_x = float(lbp_lines_df.iloc[0]['ball_x'])
sb_y = float(lbp_lines_df.iloc[0]['ball_y'])
eb_x = float(lbp_lines_df.iloc[-1]['ball_x'])
eb_y = float(lbp_lines_df.iloc[-1]['ball_y'])
ax.annotate('', xy=(eb_x, eb_y), xytext=(sb_x, sb_y),
            arrowprops=dict(arrowstyle='->, head_width=0.4, head_length=0.3',
                           color='#e6a800', lw=2.5), zorder=8)

n_broken = lbp_result.get('lines_broken_count', 0)
direction = lbp_result.get('direction_per_line', [])
ax.set_title(
    f'Line-breaking pass — {n_broken} line(s) broken — direction: {direction}',
    color='#c17f00', fontsize=11, pad=8
)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'lbp_example.png', dpi=120, bbox_inches='tight')
plt.show()

## Findings

- `label_event` produces results consistent with the precomputed annotations.
- Line-breaking passes show a clear positional progression past the representative-x threshold.
- Non-line-breaking passes stay within the defending block's depth.
- Next: see `risk_model/notebook.ipynb` to quantify expected xT by zone and game state.